LIBRARY IMPORTATION 

In [37]:
import os
import json
import pandas as pd
import numpy as np
import multiprocessing
from datetime import timedelta
import boto3 as bt3
import sagemaker as sage
from sagemaker.serializers import IdentitySerializer, JSONSerializer
from sagemaker.amazon.amazon_estimator import image_uris
from time import strftime, gmtime

IAM POLICY IDENTIFICATION AND DATA LABELLING

In [ ]:
fs_region = bt3.session.Session().region_name
fs_train = pd.read_csv('FS_V1_prep.csv')
fs_out_key= "s3://foodsecbucket/TRAIN_METRIC/model_output"
fs_train_key= "s3://foodsecbucket/TRAIN_METRIC/training"
fs_test_key= "s3://foodsecbucket/TRAIN_METRIC/testing"

In [ ]:
# MODEL PARAMETER DECLARATION
freq = 'M'
prediction_length = len(fs_train['Year'])*0.3  # 10 year
context_length = len(fs_train['Year'])*0.7
general_start = fs_train['Year'].min()
general_end = fs_train['Year'].max()

In [65]:
# MODEL ININTIALIZATION
image_name = image_uris.retrieve(framework="forecasting-deepar", region=fs_region)  # container call on global
sagemaker_session = sage.Session()
role = sage.get_execution_role()  # IAM role for sagemaker

[03/08/25 13:10:04] INFO     Same images used for training and inference. Defaulting to image     ]8;id=867557;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=96986;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#393\393]8;;\
                             scope: inference.                                                                     

[03/08/25 13:10:05] INFO     Ignoring unnecessary instance type: None.                            ]8;id=799788;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=232555;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#530\530]8;;\

MODEL CREATION

In [66]:
estimator = sage.estimator.Estimator(
    image_uri=image_name,
    sagemaker_session=sagemaker_session,
    role=role,
    instance_count=1,
    instance_type="ml.c4.2xlarge",
    base_job_name="deepar-FoodSecurity-v1",
    output_path=fs_out_key
)

# setting up Model Tunes
hyperparameters = {
    "time_freq": freq,
    "epochs": "40",
    "early_stopping_patience": "40",
    "learning_rate": "5E-4",
    "context_length": str(context_length),
    "prediction_length": str(prediction_length),
}
estimator.set_hyperparameters(**hyperparameters)

MODEL TRAINING 

In [67]:
data_channels = {"train": fs_train_key, "test": fs_test_key}
estimator.fit(inputs=data_channels, wait=False)

[03/08/25 13:10:09] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=895198;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=604146;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=537994;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=117958;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             deepar-FoodSecurity-v1-2025-03-08-13-10-09-289                                        

MODEL DEPLOYMENT

PREDICTOR CREATION

In [ ]:
"""CLASS PREDICTION CREATED TO HANDLE ROBUST FILE FORMATS AND CONFIGURATIONS"""


class DeepARPredictor(sage.predictor.Predictor):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            serializer=IdentitySerializer(content_type="application/json"),  # Ensures robust serialization
            **kwargs,
        )

    def Predict(self, fs, cat=None, dynamic_feat=None, num_samples=50, return_samples=False, quantiles=None):
        """Runs prediction using the DeepAR model with Parquet format handling."""
        if quantiles is None:
            quantiles = ["0.1", "0.5", "0.9"]

        # Ensure time series index has a frequency
        if fs.index.freq is None:
            fs.index.freq = pd.infer_freq(fs.index)
            if fs.index.freq is None:
                raise ValueError("Cannot infer time series frequency. Ensure timestamps are evenly spaced.")

        prediction_time = fs.index[-1] + fs.index.freq
        req = self.__encode_request(fs, cat, dynamic_feat, num_samples, return_samples, quantiles)

        # Make API call to SageMaker endpoint
        try:
            res = super(DeepARPredictor, self).predict(req)
        except Exception as e:
            raise RuntimeError(f"Error during prediction: {e}")

        return self.__decode_response(res, fs.index.freq, prediction_time, return_samples)

    def __encode_request(self, fs, cat, dynamic_feat, num_samples, return_samples, quantiles):
        """Encodes time series data into Parquet format and JSON metadata."""
        instance = series_to_dict(fs, cat, dynamic_feat)
        configuration = {
            "num_samples": num_samples,
            "output_types": ["quantiles", "samples"] if return_samples else ["quantiles"],
            "quantiles": quantiles,
        }

        # Convert instance data to Parquet format
        parquet_data = self.__convert_to_parquet([instance])

        # Convert metadata to JSON
        metadata = json.dumps({"configuration": configuration}).encode("utf-8")

        # Combine both into a single dictionary
        http_request_data = {"parquet_data": parquet_data, "metadata": metadata}
        return json.dumps(http_request_data).encode("utf-8")

    def __decode_response(self, response, freq, prediction_time, return_samples):
        """Decodes API response from Parquet format into a Pandas DataFrame and saves it."""
        try:
            decoded_response = json.loads(response.decode("utf-8"))
            parquet_data = decoded_response["parquet_data"]
        except (KeyError, json.JSONDecodeError) as e:
            raise ValueError(f"Invalid response format: {e}")

        # Convert Parquet back to DataFrame
        predictions = self.__convert_from_parquet(parquet_data)

        prediction_length = len(next(iter(predictions["quantiles"].values())))
        prediction_index = pd.date_range(start=prediction_time, freq=freq, periods=prediction_length)

        dict_of_samples = {
            "sample_" + str(i): s for i, s in enumerate(predictions.get("samples", []))
        } if return_samples else {}

        result_df = pd.DataFrame(data={**predictions["quantiles"], **dict_of_samples}, index=prediction_index)

        # Save as Parquet file
        result_df.to_parquet("deepAR_predictions.parquet")

        return result_df

    def __convert_to_parquet(self, data):
        """Helper function to convert JSON data to Parquet format."""
        df = pd.DataFrame(data)
        buffer = BytesIO()
        df.to_parquet(buffer, index=False)
        return buffer.getvalue().decode("ISO-8859-1")  # Convert binary data to a string for transmission

    def __convert_from_parquet(self, parquet_data):
        """Helper function to convert Parquet format back to DataFrame."""
        buffer = BytesIO(parquet_data.encode("ISO-8859-1"))  # Convert string back to binary
        return pd.read_parquet(buffer)


MODEL ENDPOINT

In [ ]:
"""ENDPOINT CREATED TO MODIFY DEPLOYMENT FUNCTION OF MODEL TO ACCOMODATE DEEPARPREDICTOR CLASS AND OTHER INITIALIZATIONS REQUIRED"""

# Deploy Model Endpoint
def deploy_model():
    estimator = sage.estimator.Estimator(
        image_uri="522234722520.dkr.ecr.us-east-1.amazonaws.com/sagemaker-deepar:latest",
        role=role,
        instance_count=1,
        instance_type="ml.m5.large",
        sagemaker_session=sagemaker_session
    )

    endpoint_name = "FoodSecurity-AR-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    predictor = estimator.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        predictor_cls=DeepARPredictor,
        endpoint_name=endpoint_name,
    )

    return predictor, endpoint_name


# Perform Inference
def run_inference(predictor, df, category=None, dynamic_features=None):
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Ensure the frequency is set correctly
    freq = pd.infer_freq(df.index)
    if freq is None:
        raise ValueError("Unable to infer frequency. Ensure the time series index is correctly formatted.")

    predictor.set_frequency(freq)

    # Run prediction
    predictions = predictor.Predict(fs=df, cat=category, dynamic_feat=dynamic_features)
    return predictions


MODEL DEPLOYMENT

In [ ]:
FS_predictor, FS_endpoint_name = deploy_model()